In [ ]:
# step6_removal_keyword_check.py

import os
import requests
import pandas as pd
from dotenv import load_dotenv
from base64 import b64decode
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found.")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === Define keyword groups ===
removal_keywords = {
    "library": ["library", "sdk", "plugin", "api", "framework", "adapter", "wrapper"],
    "demo": ["demo", "tutorial", "guide", "example", "howto"],
    "toy": ["toy", "test app", "testing", "sample app", "minimal", "playground"],
}

# === Paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step5_manifest_check_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step6_removal_keyword_check_output.csv"

# === Load CSV ===
df = pd.read_csv(input_path)

# === Prepare output column ===
removal_group_hits = []

for i, row in df.iterrows():
    if row.get("has_activity") != "yes":
        removal_group_hits.append("N/A")
        continue

    repo = row["full_name"]
    name = str(row.get("name", "")).lower()
    description = str(row.get("description", "")).lower()
    topics = str(row.get("topics", "")).lower()
    all_text = f"{name} {description} {topics}"

    # === Get README content ===
    readme_text = ""
    readme_url = f"https://api.github.com/repos/{repo}/readme"
    r = requests.get(readme_url, headers=get_headers())
    if r.status_code == 200:
        try:
            content = r.json().get("content", "")
            readme_text = b64decode(content).decode("utf-8", errors="ignore").lower()
        except:
            pass

    all_text += f" {readme_text}"

    # === Check each keyword group ===
    matched_groups = []
    for group, keywords in removal_keywords.items():
        if any(kw in all_text for kw in keywords):
            matched_groups.append(group)

    removal_group_hits.append(", ".join(matched_groups) if matched_groups else "none")

    if i % 50 == 0:
        print(f"🔎 Checked {i+1} repos...")

# === Save updated CSV ===
df["removal_keyword_group"] = removal_group_hits
df.to_csv(output_path, index=False)
print(f"✅ Step 6 complete. Saved to: {output_path}")
